# Student Persistence Model
## 03. Models

This notebook trains and compares three models that predict, by week four, which enrollments will end in withdrawal. Results are judged with precision and recall, since accuracy alone can be misleading when one outcome is more common than the other.

## Setup

In [1]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

DATA_DIR = "data"
RANDOM_STATE = 42

## Load the feature table

In [2]:
df = pd.read_csv(f"{DATA_DIR}/features.csv")
print(df.shape)
print(f"Withdrawal rate: {df['withdrew'].mean():.3f}")

(27538, 19)
Withdrawal rate: 0.186


## Choose the features

The model uses course activity and academic background. Gender, age, disability, IMD band, and region are **not** used to make predictions. They are kept aside to check fairness in the next notebook.

Two very small education categories are merged with their nearest neighbor so the model is not learning from only a few hundred rows.

In [3]:
df["education_grouped"] = df["highest_education"].replace({
    "No Formal quals": "Lower Than A Level",
    "Post Graduate Qualification": "HE Qualification",
})

numeric_features = [
    "total_clicks", "active_days", "clicks_week1", "clicks_week4",
    "pre_course_clicks", "days_since_last_active", "days_registered_early",
    "num_of_prev_attempts", "studied_credits",
]
categorical_features = ["education_grouped"]

X = df[numeric_features + categorical_features]
y = df["withdrew"]
groups = df["id_student"]

## Split into training and test sets

The split is done by student, so a student who took more than one course is entirely in training or entirely in testing. Otherwise the model could partly "recognize" students it has already seen, and results would look better than they really are.

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[test_idx])
print(f"Training: {len(X_train)} enrollments, withdrawal rate {y_train.mean():.3f}")
print(f"Test:     {len(X_test)} enrollments, withdrawal rate {y_test.mean():.3f}")

Training: 21962 enrollments, withdrawal rate 0.182
Test:     5576 enrollments, withdrawal rate 0.201


## Preprocessing

Missing registration dates are filled with the median from the training data. Numbers are scaled for logistic regression, and education level is turned into indicator columns.

In [5]:
def make_preprocessor(scale):
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(numeric_steps), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

## Models

All three models use `class_weight="balanced"`, which gives withdrawals more weight during training so the model does not simply favor the more common outcome.

In [6]:
models = {
    "Logistic regression": Pipeline([
        ("prep", make_preprocessor(scale=True)),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ]),
    "Decision tree": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("model", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random forest": Pipeline([
        ("prep", make_preprocessor(scale=False)),
        ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                         random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

## Tune the tree-based models

A small search picks how complex each tree model can be, using 5-fold cross-validation on the training data only. Folds are also split by student. F1 is used as the score because it balances precision and recall.

In [7]:
param_grids = {
    "Decision tree": {"model__max_depth": [3, 5, 8, 12]},
    "Random forest": {"model__min_samples_leaf": [5, 20, 50]},
}
cv = GroupKFold(n_splits=5)

for name, grid in param_grids.items():
    search = GridSearchCV(models[name], grid, scoring="f1", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train, groups=groups_train)
    models[name] = search.best_estimator_
    print(f"{name}: best {search.best_params_}, cross-validated F1 {search.best_score_:.3f}")

Decision tree: best {'model__max_depth': 5}, cross-validated F1 0.330
Random forest: best {'model__min_samples_leaf': 50}, cross-validated F1 0.335


## Train and evaluate

- **Precision:** of the students the model flags, how many actually withdrew.
- **Recall:** of the students who withdrew, how many the model caught.
- **AUC:** how well the model ranks students from higher to lower risk (0.5 is random, 1.0 is perfect).

A simple baseline that predicts "stays" for everyone is included for comparison.

In [8]:
rows = []
baseline_pred = pd.Series(0, index=y_test.index)
rows.append({
    "Model": "Baseline (predict stays)",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": 0.0, "Recall": 0.0, "F1": 0.0, "AUC": 0.5,
})

predictions = {}
for name, model in models.items():
    if name == "Logistic regression":
        model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = (pred, prob)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

,Accuracy,Precision,Recall,F1,AUC
Model,,,,,
Baseline (predict stays),0.799,0.000,0.000,0.000,0.500
Logistic regression,0.589,0.264,0.588,0.365,0.620
Decision tree,0.533,0.245,0.638,0.354,0.609
Random forest,0.609,0.272,0.565,0.367,0.631


## Confusion matrices

- **False positive:** a student gets a check-in they did not need. Low cost.
- **False negative:** a student who needed support was missed. Higher cost.

In [9]:
for name, (pred, _) in predictions.items():
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f"{name}")
    print(f"  Correctly flagged (true positives):   {tp}")
    print(f"  Missed (false negatives):             {fn}")
    print(f"  Flagged but stayed (false positives): {fp}")
    print(f"  Correctly not flagged:                {tn}\n")

Logistic regression
  Correctly flagged (true positives):   658
  Missed (false negatives):             461
  Flagged but stayed (false positives): 1831
  Correctly not flagged:                2626

Decision tree
  Correctly flagged (true positives):   714
  Missed (false negatives):             405
  Flagged but stayed (false positives): 2200
  Correctly not flagged:                2257

Random forest
  Correctly flagged (true positives):   632
  Missed (false negatives):             487
  Flagged but stayed (false positives): 1691
  Correctly not flagged:                2766



## Which features matter most?

For logistic regression, larger coefficients (positive or negative) mean a stronger link with withdrawal, holding the other features constant. For the random forest, importance shows how much each feature helped split the data. Neither one shows that a feature *causes* withdrawal.

In [10]:
lr = models["Logistic regression"]
feature_names = lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["model"].coef_[0], index=feature_names)
coefs.reindex(coefs.abs().sort_values(ascending=False).index).round(3)

num__studied_credits                            0.256
num__active_days                               -0.196
cat__education_grouped_Lower Than A Level       0.145
cat__education_grouped_A Level or Equivalent   -0.134
num__days_registered_early                      0.127
num__clicks_week4                              -0.122
num__total_clicks                               0.120
num__pre_course_clicks                         -0.107
cat__education_grouped_HE Qualification        -0.055
num__clicks_week1                              -0.047
num__days_since_last_active                     0.033
num__num_of_prev_attempts                       0.015
dtype: float64

In [11]:
rf = models["Random forest"]
importances = pd.Series(rf.named_steps["model"].feature_importances_,
                        index=rf.named_steps["prep"].get_feature_names_out())
importances.sort_values(ascending=False).round(3)

num__studied_credits                            0.173
num__total_clicks                               0.134
num__days_registered_early                      0.119
num__clicks_week1                               0.111
num__pre_course_clicks                          0.106
num__clicks_week4                               0.105
num__active_days                                0.103
num__days_since_last_active                     0.061
cat__education_grouped_Lower Than A Level       0.044
cat__education_grouped_A Level or Equivalent    0.019
num__num_of_prev_attempts                       0.013
cat__education_grouped_HE Qualification         0.011
dtype: float64

## Save test predictions for the fairness check

The next notebook compares how each model performs for different student groups.

In [12]:
fairness_cols = ["gender", "age_band", "disability", "imd_band", "highest_education"]
test_out = df.iloc[test_idx][["id_student"] + fairness_cols + ["withdrew"]].copy()
for name, (pred, prob) in predictions.items():
    key = name.lower().replace(" ", "_")
    test_out[f"{key}_pred"] = pred
    test_out[f"{key}_prob"] = prob

test_out.to_csv(f"{DATA_DIR}/test_predictions.csv", index=False)
print(test_out.shape)

(5576, 13)


## Summary

_Fill in after running:_

- Best model and why: 
- Precision and recall of the best model: 
- Most important features: 
- What a false negative and a false positive would mean for students: 